## CASCADE Conformal Prediction: Uncertainty-Adaptive Prediction Intervals for Two-Stage Clinical Decision Support

Official Implementation of "CASCADE Conformal Prediction: Uncertainty-Adaptive Prediction Intervals for Two-Stage Clinical Decision Support" (Diaz-Rincon et al., 2026).

(Diaz-Rincon et al., 2025) used the predicted probability of LEDD change for prediction of medication needs in Parkinson's Disease patients in a Two-Stage ensamble without uncertainty propagation.

Our current approach calculates the patient's epistemic uncertainty in Stage 1 through Venn Abers (VA) uncertainty to inform Stage 2 Conformal Prediction (CP) intervals.

Findings:
- Low uncertainty → Stage 2 provides narrow intervals (confident)
- High uncertainty → Stage 2 provides wide intervals (cautious)

This is the CASCADE effect: Stage 1 confidence propagates to Stage 2, adapting the prediction intervals.

In [1]:
!pip install venn-abers
!pip install mapie==0.8.6

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from venn_abers import VennAbers

from mapie.regression import MapieRegressor
from mapie.metrics import regression_coverage_score, regression_mean_width_score
from mapie.subsample import Subsample

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, mean_absolute_error, r2_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestRegressor

from scipy import stats
from scipy.stats import ttest_1samp, bootstrap

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

In [3]:
# Global Configuration
RANDOM_STATE = 21
ALPHA = 0.2 # Targetting 80% Coverage
USE_PREDICTED_STAGE1 = False
STAGE1_MODEL_TYPE = "XGBoost"
Q_hat = 0

# Conformal Methods
STRATEGIES = {
    "naive": {"method": "naive"},
    "split": {'method': 'base'},
    "cv_plus": {"method": "plus", "cv": 10},
    "jackknife_plus_ab": {"method": "plus", "cv": Subsample(n_resamplings=50)},
}

# XGBoost Parameters
XGBOOST_BINARY_PARAMS = {
    "eval_metric": "auc",
    "objective": "binary:logistic",
    'sampling_method': 'gradient_based',
    'alpha': 0.1,
    'lambda': 1,
    'learning_rate': 0.1,
    'max_depth': 7,
    'tree_method': 'hist',
    'device': "cuda",
}

XGBOOST_REGRESSION_PARAMS = {
    'alpha': 0.1,
    'lambda': 1,
    'learning_rate': 0.1,
    'max_depth': 7,
    'n_estimators': 700,
    "eval_metric": 'rmse',
    'objective': 'reg:squarederror',
    'sampling_method': 'gradient_based',
    'tree_method': 'hist',
    'device': "cuda"
}

# Colors used for figures
color_azure = '#0097B2'
color_orange = '#F35000'
color_red = '#B00000'
color_gold = '#FFD700'

## Data Loading and Preprocessing

In [4]:
path = 'data/data_1Y.csv'
data = pd.read_csv(path)

# Continous Outcome (LEDD Change)
normalized_pctg_change = data['normalized_percent_change']  # Save for Stage 2
data.drop(columns=['normalized_percent_change'], inplace=True)

# Prepare features
xgboost_df = data.copy()

# One-hot encoding
demographic_vars = ['gender_source_value', 'race_source_value', 'ethnicity_source_value']
xgboost_df = pd.get_dummies(xgboost_df, columns=demographic_vars)

# Scaling
scaler = MinMaxScaler()
numeric_vars = ['mean_led_per_visit', 'age', 'length_of_stay', 'days_since_last_visit', 'days_to_diagnosis']
for var in numeric_vars:
    xgboost_df[var] = scaler.fit_transform(xgboost_df[[var]])

# Reordering columns
prediction_to_last = xgboost_df.pop('prediction')
xgboost_df['prediction'] = prediction_to_last

# Features and targets
X = xgboost_df.iloc[:, :-1]
y_binary = xgboost_df.iloc[:, -1]  # Binary: medication change needed
y_continuous = normalized_pctg_change  # Continuous: percent change in LEDD

# Stage 1: Binary Classification with Venn-Abers
Get VA uncertainty estimates for ALL patients in Stage 1 binary prediction. This will inform Stage 2 interval lengths

In [5]:
# Split for Stage 1: Train on 80%, use 20% for VA calibration
X_train_stage1, X_test_stage1, y_train_stage1, y_test_stage1 = train_test_split(
    X, y_binary, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Stage 1 Split | Training: {len(X_train_stage1)} ({len(X_train_stage1)/len(X)*100:.1f}%) | VA Calibration: {len(X_test_stage1)} ({len(X_test_stage1)/len(X)*100:.1f}%)")
print(f"\nTraining Stage 1 {STAGE1_MODEL_TYPE} classifier...")

if STAGE1_MODEL_TYPE == "XGBoost":
    # Train XGBoost binary classifier on FULL training set
    d_train = xgb.DMatrix(X_train_stage1, label=y_train_stage1)
    d_test = xgb.DMatrix(X_test_stage1, label=y_test_stage1)
    d_all = xgb.DMatrix(X)

    model_stage1 = xgb.train(XGBOOST_BINARY_PARAMS, d_train,num_boost_round=700,
        evals=[(d_test, "validation")], verbose_eval=100,early_stopping_rounds=10
    )

    # Getting uncalibrated probabilities 
    p_test_uncalibrated = model_stage1.predict(d_test, iteration_range=(0, model_stage1.best_iteration + 1))
    p_all_uncalibrated = model_stage1.predict(d_all,iteration_range=(0, model_stage1.best_iteration + 1))

else:
    # Train Logistic Regression for Stage 1
    model_stage1 = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    model_stage1.fit(X_train_stage1, y_train_stage1)
    
    # Get uncalibrated probabilities (extracting class 1 probability)
    p_test_uncalibrated = model_stage1.predict_proba(X_test_stage1)[:, 1]
    p_all_uncalibrated = model_stage1.predict_proba(X)[:, 1]

# Venn-Abers Calibration
print("Calibrating with Venn-Abers on held-out test set...")
p_test_2d = np.column_stack([1 - p_test_uncalibrated, p_test_uncalibrated]) # Convert to 2D for VennAbers library

va = VennAbers()
va.fit(p_test_2d, y_test_stage1.values)

# VA Uncertainty Scores for full cohort
print("Computing VA uncertainty scores for the full patient cohort...")
p_all_2d = np.column_stack([1 - p_all_uncalibrated, p_all_uncalibrated]) # Convert to 2D and get VA intervals


p_lower_all_2d, p_upper_all_2d = va.predict_proba(p_all_2d)

# Extract predictions for class 1 (needs medication change)
p_lower_all = p_lower_all_2d[:, 1]
p_upper_all = p_upper_all_2d[:, 1]
va_uncertainty_all = p_upper_all - p_lower_all

print(f"VA Uncertainty Stats (n={len(X)}): Mean={va_uncertainty_all.mean():.4f}, Median={np.median(va_uncertainty_all):.4f}, Max={va_uncertainty_all.max():.4f}")

Stage 1 Split | Training: 4884 (80.0%) | VA Calibration: 1221 (20.0%)

Training Stage 1 XGBoost classifier...
[0]	validation-auc:0.86294
[100]	validation-auc:0.95154
[200]	validation-auc:0.96313
[242]	validation-auc:0.96421
Calibrating with Venn-Abers on held-out test set...
Computing VA uncertainty scores for the full patient cohort...
VA Uncertainty Stats (n=6105): Mean=0.0069, Median=0.0003, Max=0.1537


# Selection of Stage 2 patients

We select patients in two ways: By focusing on patients who actually need medication change (y_binary == 1) or by using the predicted probability (y_hat == 1)

In [6]:
if USE_PREDICTED_STAGE1:
    print("Selection criterion: y_hat == 1 (Predicted Positives: Predicted to need medication change)")
    # Use only patients who actually need medication change. No probability filtering
    # Calculate Youden's Index on Stage 1 Test Set to find optimal threshold
    fpr, tpr, thresholds = roc_curve(y_test_stage1, p_test_uncalibrated)
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[optimal_idx]
    print(f"Optimal Classification Threshold (Youden's J): {optimal_threshold:.4f}")
    # Apply threshold to get predicted classes for the full cohort
    y_hat_stage1 = (p_all_uncalibrated > optimal_threshold).astype(int)
    mask = (y_hat_stage1 == 1)

else:
    print("Selection criterion: y_binary == 1 (True Positives: Actually needs medication change)")
    mask = (y_binary == 1)

X_stage2 = X[mask]
y_stage2 = y_continuous[mask] #Previously saved variable that contains the normalized LEDD change
va_uncertainty_stage2 = va_uncertainty_all[mask]

# Get VA uncertainty for the selected patients.
p_lower_stage2 = p_lower_all[mask]
p_upper_stage2 = p_upper_all[mask]

print(f"Patients selected: {len(X_stage2)} / {len(X)} ({len(X_stage2)/len(X)*100:.1f}%)")
print(f"  Mean: {va_uncertainty_stage2.mean():.4f} | Std: {va_uncertainty_stage2.std():.4f} | Range: [{va_uncertainty_stage2.min():.4f}, {va_uncertainty_stage2.max():.4f}]")

Selection criterion: y_binary == 1 (True Positives: Actually needs medication change)
Patients selected: 1533 / 6105 (25.1%)
  Mean: 0.0219 | Std: 0.0256 | Range: [0.0002, 0.1537]


### Stage 2 Split

In [7]:
X_train_2, X_test_2, y_train_2, y_test_2, va_unc_train_2, va_unc_test_2, p_lower_train_2, p_lower_test_2 = train_test_split(X_stage2, y_stage2,va_uncertainty_stage2, p_lower_stage2,test_size=0.2, random_state=RANDOM_STATE)

print(f"Training: {len(X_train_2)} patients")
print(f"Test:     {len(X_test_2)} patients")
print(f"Mean LEDD change (train): {y_train_2.mean():.3f}")
print(f"Mean LEDD change (test):  {y_test_2.mean():.3f}")

# Uses XGBoost or Linear Regression
if STAGE1_MODEL_TYPE == "XGBoost":
    print(f"\nTraining base {STAGE1_MODEL_TYPE} regression model...")
    xgb_model_base = xgb.XGBRegressor(**XGBOOST_REGRESSION_PARAMS, random_state=RANDOM_STATE)

else:
    print(f"\nTraining base Linear Regression regression model...")
    xgb_model_base = LinearRegression()

xgb_model_base.fit(X_train_2, y_train_2)
y_pred_base = xgb_model_base.predict(X_test_2)
rmse = np.sqrt(mean_squared_error(y_test_2, y_pred_base))
mae = mean_absolute_error(y_test_2, y_pred_base)
r2 = r2_score(y_test_2, y_pred_base)

print(f"Base Model Performance:")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  R²:   {r2:.4f}")

Training: 1226 patients
Test:     307 patients
Mean LEDD change (train): 0.017
Mean LEDD change (test):  0.036

Training base XGBoost regression model...
Base Model Performance:
  RMSE: 0.0995
  MAE:  0.0380
  R²:   0.9801


In [8]:
# Helper function to calculate metrics (coverage, length and cascade ratio)
def calculate_metrics(y_true, y_lower, y_upper, uncertainties, k=3):
    """
    Calculates Coverage, Average Length, and the Cascade Ratio.
    
    Parameters:
    - k: The number of bins to divide the uncertainty space into. 
         k=3 (Tertiles) compares the top 33.3% vs bottom 33.3%.
         k=5 (Quintiles) compares the top 20% vs bottom 20%.
         k=7 (Septiles) compares the top 14.28% vs bottom 14.28%
    """
    y_true = np.asarray(y_true).flatten()
    y_lower = np.asarray(y_lower).flatten()
    y_upper = np.asarray(y_upper).flatten()
    u_test = np.asarray(uncertainties).flatten()
    
    # Coverage
    covered = (y_true >= y_lower) & (y_true <= y_upper)
    cov = np.mean(covered)

    # Width
    widths = y_upper - y_lower
    avg_width = np.mean(widths)

    # Cascade Ratio (Generalized for K groups)
    p_low = 100.0 / k
    p_high = 100.0 * (k - 1) / k
    
    t_low = np.percentile(va_unc_train_2, p_low)
    t_high = np.percentile(va_unc_train_2, p_high)

    mask_low = uncertainties <= t_low
    mask_high = uncertainties > t_high

    w_low = np.mean(widths[mask_low])
    w_high = np.mean(widths[mask_high])

    # Avoid division by zero
    ratio = w_high / w_low if w_low > 1e-6 else 0.0
    
    return cov, avg_width, ratio

# Experiment 1: Baseline Conformal Predicion (No VA Uncertainty)

In [9]:
results_baseline = {}

# Convert to numpy arrays for our unified metric function
y_true_arr = np.array(y_test_2) # Ground truth converted to numpy array
unc_arr = np.array(va_unc_test_2) # VA uncertainty for the test set, converted to numpy array

print(f"{'Strategy':<22} | {'Coverage':<10} | {'Avg Length':<10} | {'Cascade Ratio':<15}")
print("-" * 65)

for strategy, params in STRATEGIES.items():
    # We use MAPIE to get the baselines for CP methods
    mapie = MapieRegressor(
        xgb_model_base,
        test_size=0.2,
        **params,
        random_state=RANDOM_STATE
    )
    mapie.fit(X_train_2, y_train_2)
    y_pred, y_pis = mapie.predict(X_test_2, alpha=ALPHA)

    # Extract conformal bounds
    y_lower = y_pis[:, 0, 0]
    y_upper = y_pis[:, 1, 0]

    # Evaluate using our helper function
    coverage, width, ratio = calculate_metrics(y_true_arr, y_lower, y_upper, unc_arr)

    # Store results 
    results_baseline[strategy] = {
        'y_pred': y_pred,
        'y_lower': y_lower,
        'y_upper': y_upper,
        'coverage': coverage,
        'width': width,
        'ratio': ratio
    }

    print(f"{strategy:<22} | {coverage:<10.3f} | {width:<10.3f} | {ratio:<15.2f}")

Strategy               | Coverage   | Avg Length | Cascade Ratio  
-----------------------------------------------------------------
naive                  | 0.524      | 0.031      | 1.00           
split                  | 0.840      | 0.113      | 1.00           
cv_plus                | 0.834      | 0.100      | 1.06           
jackknife_plus_ab      | 0.635      | 0.132      | 0.99           


# Experiment 2: Stratified by VA Uncertainty (Mondrian CP)

Here we separate per bins (k) based on uncertainty quantiles from the VA estimates

In [ ]:
K_BINS = 3  # Dynamic parameter: Creates K equal-sized uncertainty bins

print(f"Stratified VA (Mondrian CP) for K={K_BINS}")

# Dynamically calculate percentiles and edges
percentiles = np.linspace(0, 100, K_BINS + 1)
bin_edges = np.percentile(va_unc_train_2, percentiles)
bin_labels = [str(i) for i in range(1, K_BINS + 1)]

print(f"\nCreated {K_BINS} Uncertainty Bins (from training set):")
for i in range(K_BINS):
    print(f"  Bin {i+1}: [{bin_edges[i]:.4f}, {bin_edges[i+1]:.4f}]")

# Apply dynamic bins to Train and Test sets
bins_train = pd.cut(va_unc_train_2, bins=bin_edges, labels=bin_labels, include_lowest=True, duplicates='drop')
bins_test = pd.cut(va_unc_test_2, bins=bin_edges, labels=bin_labels, include_lowest=True, duplicates='drop')

print(f"\nBin Distribution (Test Set):")
for cat in bin_labels:
    n_test_cat = (bins_test == cat).sum()
    print(f"  Bin {cat}: {n_test_cat:4d} samples")

results_stratified = {}
y_true_arr = np.array(y_test_2)
unc_arr = np.array(va_unc_test_2)

for strategy, params in STRATEGIES.items():
    print(f"\n--- Running Stratified: {strategy} ---")

    y_pred_all = np.zeros(len(X_test_2))
    y_lower_all = np.zeros(len(X_test_2))
    y_upper_all = np.zeros(len(X_test_2))
    
    # Process each dynamic bin separately
    for category in bin_labels:
        mask_train = (bins_train == category)
        mask_test = (bins_test == category)

        n_train_cat = mask_train.sum()
        n_test_cat = mask_test.sum()
        
        # Skip the bin if data is insuficient
        if n_train_cat < 10 or n_test_cat < 2:
            print(f"  Bin {category}: SKIPPED (insufficient data - Fragmentation Issue!)")
            continue

        X_train_cat, y_train_cat = X_train_2[mask_train], y_train_2[mask_train]
        X_test_cat = X_test_2[mask_test]
        y_test_cat = y_true_arr[mask_test]
        
        # Added after reviews: Toggle to use XGBoost or Linear Regression
        if STAGE1_MODEL_TYPE == "XGBoost":
            model_cat = xgb.XGBRegressor(**XGBOOST_REGRESSION_PARAMS, random_state=RANDOM_STATE)
        else:
            model_cat = LinearRegression()

        model_cat.fit(X_train_cat, y_train_cat)

        mapie_cat = MapieRegressor(model_cat, test_size=0.2, **params, random_state=RANDOM_STATE)
        mapie_cat.fit(X_train_cat, y_train_cat)
        
        y_pred_cat, y_pis_cat = mapie_cat.predict(X_test_cat, alpha=ALPHA)

        y_pred_all[mask_test] = y_pred_cat
        y_lower_all[mask_test] = y_pis_cat[:, 0, 0]
        y_upper_all[mask_test] = y_pis_cat[:, 1, 0]
        
        # Local Bin Length for logging
        local_width = np.mean(y_pis_cat[:, 1, 0] - y_pis_cat[:, 0, 0])
        local_cov = np.mean((y_test_cat >= y_pis_cat[:, 0, 0]) & (y_test_cat <= y_pis_cat[:, 1, 0]))
        print(f"  Bin {category}: Cov = {local_cov*100:.1f}% | Interval Length = {local_width:.3f} (n_test={n_test_cat})")

    # Global Evaluation using helper function
    cov, wid, rat = calculate_metrics(y_true_arr, y_lower_all, y_upper_all, unc_arr, k=K_BINS)

    results_stratified[strategy] = {
        'y_pred': y_pred_all, 'y_lower': y_lower_all, 'y_upper': y_upper_all,
        'coverage': cov, 'width': wid, 'ratio': rat, 'method': 'stratified'
    }

    print(f"\n  OVERALL {strategy} | Coverage: {cov:.3f} | Width: {wid:.3f} | CR: {rat:.2f}")

Stratified VA (Mondrian CP) for K=3

Created 3 Uncertainty Bins (from training set):
  Bin 1: [0.0002, 0.0077]
  Bin 2: [0.0077, 0.0189]
  Bin 3: [0.0189, 0.1537]

Bin Distribution (Test Set):
  Bin 1:  122 samples
  Bin 2:   89 samples
  Bin 3:   96 samples

--- Running Stratified: naive ---
  Bin 1: Cov = 58.2% | Interval Length = 0.031 (n_test=122)
  Bin 2: Cov = 60.7% | Interval Length = 0.019 (n_test=89)
  Bin 3: Cov = 63.5% | Interval Length = 0.028 (n_test=96)

  OVERALL naive | Coverage: 0.606 | Width: 0.027 | CR: 0.90

--- Running Stratified: split ---
  Bin 1: Cov = 80.3% | Interval Length = 0.090 (n_test=122)
  Bin 2: Cov = 88.8% | Interval Length = 0.090 (n_test=89)
  Bin 3: Cov = 92.7% | Interval Length = 0.181 (n_test=96)

  OVERALL split | Coverage: 0.866 | Width: 0.118 | CR: 2.02

--- Running Stratified: cv_plus ---
  Bin 1: Cov = 82.8% | Interval Length = 0.094 (n_test=122)
  Bin 2: Cov = 85.4% | Interval Length = 0.106 (n_test=89)
  Bin 3: Cov = 93.8% | Interval Lengt

In [ ]:
print("How Stage 1 VA Uncertainty Propagates to Stage 2 Intervals:")
# We loop through the Mondrian (stratified) results we stored in the previous experiment
for strategy in STRATEGIES.keys():
    if strategy not in results_stratified:
        continue
        
    print(f"\n{strategy.upper()}:")
    
    # Retrieve the global arrays for this CP model
    y_lower_all = results_stratified[strategy]['y_lower']
    y_upper_all = results_stratified[strategy]['y_upper']
    
    # Iterate through our K_BINS
    for category in bin_labels:
        mask = (bins_test == category)
        n_test = mask.sum()
        
        # Skip if the bin is empty
        if n_test == 0:
            continue
            
        unc_range = va_unc_test_2[mask]
        
        # Calculate Local Metrics for this specific bin
        y_test_cat = y_test_2[mask]
        y_lower_cat = y_lower_all[mask]
        y_upper_cat = y_upper_all[mask]
        
        cov_cat = np.mean((y_test_cat >= y_lower_cat) & (y_test_cat <= y_upper_cat))
        width_cat = np.mean(y_upper_cat - y_lower_cat)
        
        print(f"  Bin {category:2s} | VA Uncertainty: [{unc_range.min():.4f}, {unc_range.max():.4f}]")
        print(f"         → Stage 2 Avg Length: {width_cat:.4f}")
        print(f"         → Local Coverage:    {cov_cat:.4f} (n={n_test})")

In [ ]:
# Split Training Data into "Proper Train" and "Calibration" sets
# We need a fresh set of data to calibrate the scores, otherwise residuals are too small.
X_train_proper, X_calib, y_train_proper, y_calib, va_unc_proper, va_unc_calib = train_test_split(
    X_train_2, y_train_2, va_unc_train_2,
    test_size=0.2,
    random_state=42
)

print(f"Split Statistics:")
print(f"  Proper Training: {len(X_train_proper)} samples")
print(f"  Calibration:     {len(X_calib)} samples")
print(f"  Test:            {len(X_test_2)} samples")

# Retrain Base Model on Proper Training Set ONLY
print("\nRetraining base model on Proper Training set...")
if STAGE1_MODEL_TYPE == "XGBoost":
    print(f"Using XGBoost for Stage 2 regression...")
    xgb_model_proper = xgb.XGBRegressor(**XGBOOST_REGRESSION_PARAMS, random_state=RANDOM_STATE)
else:
    print(f"Using Linear Regression for Stage 2 regression...")
    xgb_model_proper = LinearRegression()
    
xgb_model_proper.fit(X_train_proper, y_train_proper)

# Experiment 3: Continous Cascade (Mean-Centered)

In [ ]:
print(f" Continous (Mean-Centered) CP (Target: {1-ALPHA:.0%}) ---")
def get_scaling_factor_centered(va_uncertainty, global_mean_unc, beta=0.5, min_scale=0.2, max_scale=5.0):
    norm_diff = (va_uncertainty / global_mean_unc) - 1.0
    sigma = 1.0 + (beta * norm_diff)
    sigma = np.clip(sigma, a_min=min_scale, a_max=max_scale)
    return sigma

# Setup
betas = np.arange(0.0, 1.6, 0.1)
n_boot = 1000
y_true = np.array(y_test_2)
u_test = np.array(va_unc_test_2)
global_mean_unc = np.mean(va_unc_proper)
K_BINS = 3

# Target-Seeking Tracker Variables
best_coverage_error = 1.0   # We want to minimize |actual_cov - target_cov|
opt_beta = 0.0
best_ratio_of_opt = 1.0     # Stores the CR of the selected beta
final_y_lower = None
final_y_upper = None

print(f"{'Beta':<5} | {'Cov %':<8} | {'Len':<6} | {'CR':<6} | {'P-Value':<8} | {'Status'}")
print("-" * 65)

sensitivity_results = []
w_cv = results_baseline['cv_plus']['y_upper'] - results_baseline['cv_plus']['y_lower']

# Pre-calculate Base Model predictions to save time
y_pred_calib_base = xgb_model_proper.predict(X_calib)
residuals_calib_base = np.abs(y_calib - y_pred_calib_base)
y_pred_test_base = xgb_model_proper.predict(X_test_2)

# Sensitivity (Beta) Loop
for b in betas:
    # Recalibrate
    sigma_calib = get_scaling_factor_centered(va_unc_calib, global_mean_unc, beta=b)
    scores = residuals_calib_base / sigma_calib

    n_calib = len(scores)
    q_level = np.ceil((n_calib + 1) * (1 - ALPHA)) / n_calib
    q_hat = np.quantile(scores, min(1.0, q_level), method='higher')
    Q_hat = q_hat

    # Test
    sigma_test = get_scaling_factor_centered(u_test, global_mean_unc, beta=b)
    half_width = q_hat * sigma_test
    y_lower = y_pred_test_base - half_width
    y_upper = y_pred_test_base + half_width
    widths = y_upper - y_lower

    # Getting Metrics for a specific bin
    cov, avg_len, ratio = calculate_metrics(y_true, y_lower, y_upper, u_test, k=K_BINS)
    
    # Bootstrap vs Split CP Baseline) to get p-values
    diffs = []
    for i in range(n_boot):
        idx = resample(np.arange(len(y_true)), replace=True)
        # Re-calc ratio for this bootstrap sample
        _, _, r_b = calculate_metrics(y_true[idx], y_lower[idx], y_upper[idx], u_test[idx], k=K_BINS)
        _, _, r_base = calculate_metrics(y_true[idx], results_baseline['split']['y_lower'][idx], results_baseline['split']['y_upper'][idx], u_test[idx], k=K_BINS)
        diffs.append(r_b - r_base)

    p_val = np.mean(np.array(diffs) <= 0)

    # Status and Tracking
    status = "Valid" if cov >= (1 - ALPHA) else "Invalid"
    if cov >= (1 - ALPHA + 0.05): status = "Conservative"

    sensitivity_results.append({
        'beta': b, 'cov': cov, 'len': avg_len, 'ratio': ratio, 'p_val': p_val, 'status': status
    })

    # We prioritize the beta that hits the target coverage most accurately (minimizing error)
    cov_error = abs(cov - (1 - ALPHA))
    
    if cov >= (1 - ALPHA) and b <= 1.0: # Stability constraint
        if cov_error < best_coverage_error:
            best_coverage_error = cov_error
            opt_beta = b
            best_ratio_of_opt = ratio
            final_y_lower = y_lower.copy() 
            final_y_upper = y_upper.copy()

    p_str = "<.001" if p_val < 0.001 else f"{p_val:.3f}"
    print(f"{b:<5.1f} | {cov:<8.2%} | {avg_len:<6.4f} | {ratio:<6.2f} | {p_str:<8} | {status}")

# Store results for later    
df_sens = pd.DataFrame(sensitivity_results)

print("-" * 65)
print(f"Dynamically Selected Optimal Beta: {opt_beta:.1f}")
print(f"Resulting Cascade Ratio: {best_ratio_of_opt:.2f}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(6, 5))

# Plotting Coverage (Left Axis)
ax1.set_xlabel(r'Sensitivity Parameter ($\beta$)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Marginal Coverage', color=color_azure, fontsize=11, fontweight='bold')

# Main Coverage Line
ax1.plot(df_sens['beta'], df_sens['cov'], color=color_azure, marker='o', linewidth=2, label='Coverage')

# Target Line
ax1.axhline(0.80, color=color_red, linestyle='--', alpha=0.5, label='Target (0.80)')

ax1.tick_params(axis='y', labelcolor='black')
ax1.set_ylim(0.75, 0.95)
ax1.set_xticks(df_sens['beta'])

# Shade Invalid Regions
ax1.axvspan(0.25, 0.35, color='red', alpha=0.1, label='Invalid Region')
ax1.axvspan(0.85, 1.05, color='red', alpha=0.1)
ax1.axvspan(1.45, 1.55, color='red', alpha=0.1)

# Plot Ratio (Right Axis)
ax2 = ax1.twinx()
ax2.set_ylabel('Cascade Ratio', color=color_orange, fontsize=11, fontweight='bold')
ax2.plot(df_sens['beta'], df_sens['ratio'], color=color_orange, marker='s', linewidth=2, linestyle='-.', label='Adaptivity')
ax2.tick_params(axis='y', labelcolor="black")

# Mark Optimal Point
opt_row = df_sens[np.isclose(df_sens['beta'], 0.7)].iloc[0]
ax2.plot(0.7, opt_row['ratio'], marker='*', color=color_gold, markersize=20, markeredgecolor='black', label='Optimal (0.7)')

# Legend
# Get all handles/labels from both axes
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
lines = lines_1 + lines_2
labels = labels_1 + labels_2
unique = dict(zip(labels, lines))

ordered_labels = ['Coverage', 'Target (0.80)', 'Invalid Region', 'Adaptivity', 'Optimal (0.7)']

try:
    ordered_handles = [unique[l] for l in ordered_labels]
except KeyError as e:
    print(f"Error: Label {e} not found in plot. Available labels: {list(unique.keys())}")
    ordered_handles = list(unique.values())
    ordered_labels = list(unique.keys())

# D. Plot Legend
ax1.legend(ordered_handles, ordered_labels,
           loc='lower center',
           bbox_to_anchor=(0.5, 0.995),  # Just above the plot
           ncol=5,                      # All 5 in one row
           frameon=False,
           fontsize=8,                  # Smaller font to fit 5 items
           columnspacing=0.8,           # Tight spacing
           handletextpad=0.3)

plt.grid(True, alpha=0.3)
plt.tight_layout()
# plt.savefig('sensitivity_plot.svg', dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
# High Winkler, Marginal Cov

class CascadeEvaluator:
    def __init__(self, alpha=0.2):
        self.alpha = alpha

    def get_metrics(self, y_true, y_pred, y_lower, y_upper):
        y_true, y_pred = np.asarray(y_true).flatten(), np.asarray(y_pred).flatten()
        y_low, y_high = np.asarray(y_lower).flatten(), np.asarray(y_upper).flatten()
        
        widths = y_high - y_low
        errors = np.abs(y_true - y_pred)
        
        coverage = np.mean((y_true >= y_low) & (y_true <= y_high))
        # Winkler Score
        winkler = np.mean(widths + (2.0/self.alpha) * (np.maximum(y_low - y_true, 0) + np.maximum(y_true - y_high, 0)))
        
        # Spearman Correlation for Adaptivity
        corr, _ = stats.spearmanr(widths, errors)
        if np.isnan(corr): corr = 0.0 
            
        return {
            'Winkler': float(winkler),
            'Adaptivity (Corr)': float(corr),
            'Coverage': float(coverage),        
        }

# 1. Continous Cascade Calibration

# We use the explicit locally adaptive conformal prediction math.
results_cascade = {}

# Get baseline predictions and calibration residuals
y_pred_calib = xgb_model_proper.predict(X_calib)
residuals_calib = np.abs(y_calib - y_pred_calib)
y_pred_test = xgb_model_proper.predict(X_test_2)

# Calculate continuous scaling factors
# The optimal parameter locked in from tuning
best_beta = opt_beta
sigma_calib = get_scaling_factor_centered(va_unc_calib, global_mean_unc, beta=best_beta)
sigma_test = get_scaling_factor_centered(va_unc_test_2, global_mean_unc, beta=best_beta)

# The Conformal Math
# Scale non-conformity scores
scores = residuals_calib / sigma_calib

# Calculate the global Conformal Quantile (q_hat) with finite-sample correction
n_calib = len(scores)
q_level = np.ceil((n_calib + 1) * (1 - ALPHA)) / n_calib
q_hat = np.quantile(scores, min(1.0, q_level), method='higher')

# Apply q_hat and local test uncertainty to create final bounds
half_width_test = q_hat * sigma_test

# Store the Results
results_cascade['split'] = {
    'y_pred': y_pred_test,
    'y_lower': y_pred_test - half_width_test,
    'y_upper': y_pred_test + half_width_test,
    'q_hat': q_hat
}

print(f" CASCADE Calibration finished (q_hat = {q_hat:.4f})")

# Initialize unified evaluator
evaluator = CascadeEvaluator(alpha=ALPHA)
master_results = []

# Evaluate all available Baselines
baseline_strategies = ['split', 'cv_plus', 'jackknife_plus_ab']
for strategy in baseline_strategies:
    if strategy in results_baseline:
        m = evaluator.get_metrics(
            y_test_2, 
            results_baseline[strategy]['y_pred'], 
            results_baseline[strategy]['y_lower'], 
            results_baseline[strategy]['y_upper']
        )
        master_results.append({
            'Method': f"Baseline CP ({strategy})",
            'Coverage': m['Coverage'],
            'Winkler Score ↓': m['Winkler'],
            'Adaptivity (Spearman) ↑': m['Adaptivity (Corr)']
        })

# Evaluate Continuous CASCADE (Ours)
if 'split' in results_cascade:
    m_casc = evaluator.get_metrics(
        y_test_2, 
        results_cascade['split']['y_pred'], 
        results_cascade['split']['y_lower'], 
        results_cascade['split']['y_upper']
    )
    master_results.append({
        'Method': "Continuous CASCADE (Ours)",
        'Coverage': m_casc['Coverage'],
        'Winkler Score ↓': m_casc['Winkler'],
        'Adaptivity (Spearman) ↑': m_casc['Adaptivity (Corr)']
    })

# Save results
df_master = pd.DataFrame(master_results)

# Create a clean display mapping
format_dict = {
    'Coverage': '{:.2%}',
    'Winkler Score ↓': '{:.4f}',
    'Adaptivity (Spearman) ↑': '{:.4f}'
}

print("\nTABLE 1: Global Performance Comparison")
df_master

In [ ]:
print("Stratified Performance on Top 20% and Bottom 80%")

# We use the cleaned 'y_lower' and 'y_upper' from the unified dictionaries
df_analysis = pd.DataFrame({
    'y_true': np.array(y_test_2).flatten(),
    'uncertainty': np.array(va_unc_test_2).flatten(),
    
    # Baseline Bounds (Standard CP)
    'base_low': results_baseline['split']['y_lower'].flatten(),
    'base_high': results_baseline['split']['y_upper'].flatten(),
    
    # CASCADE Bounds (Continuous Adaptive)
    'casc_low': results_cascade['split']['y_lower'].flatten(),
    'casc_high': results_cascade['split']['y_upper'].flatten()
})

# Pre-calculate widths and coverage booleans
df_analysis['base_width'] = df_analysis['base_high'] - df_analysis['base_low']
df_analysis['casc_width'] = df_analysis['casc_high'] - df_analysis['casc_low']

df_analysis['base_cov'] = (df_analysis['y_true'] >= df_analysis['base_low']) & (df_analysis['y_true'] <= df_analysis['base_high'])
df_analysis['casc_cov'] = (df_analysis['y_true'] >= df_analysis['casc_low']) & (df_analysis['y_true'] <= df_analysis['casc_high'])

# 2. Split into High/Low Risk (Top 20% vs Bottom 80%)
threshold_high = df_analysis['uncertainty'].quantile(0.80)
low_risk = df_analysis[df_analysis['uncertainty'] <= threshold_high]
high_risk = df_analysis[df_analysis['uncertainty'] > threshold_high]

# 3. Helper function for clean printing
def print_group_stats(group_name, df_sub):
    cb, cc = df_sub['base_cov'].mean(), df_sub['casc_cov'].mean()
    wb, wc = df_sub['base_width'].mean(), df_sub['casc_width'].mean()
    
    print(f"\n[{group_name} UNCERTAINTY GROUP] (N={len(df_sub)})")
    print(f"  Coverage:  Baseline = {cb:.1%} | CASCADE = {cc:.1%} (Gain: {cc-cb:+.1%})")
    print(f"  Avg Width: Baseline = {wb:.3f} | CASCADE = {wc:.3f} (Expansion: {(wc-wb)/wb:+.1%})")

# 4. Final Output
print(f"Threshold > {threshold_high:.3f} (Top 20% Uncertainty)")
print_group_stats("LOW", low_risk)
print_group_stats("HIGH", high_risk)

# Statistical Comparison
Winkler Score, KS and Spearman Rank Correlation

In [ ]:
# Winkler Score (Absolute Delta)
def winkler_calc(y, low, high, alpha=0.2):
    w = high - low
    # Penalty: (2/alpha) * distance from interval
    p = (2/alpha) * (np.maximum(low - y, 0) + np.maximum(y - high, 0))
    return np.mean(w + p)

w_base = winkler_calc(df_analysis.y_true, df_analysis.base_low, df_analysis.base_high)
w_casc = winkler_calc(df_analysis.y_true, df_analysis.casc_low, df_analysis.casc_high)
w_delta = w_casc - w_base

print(f"   Winkler Scores")
print(f"   Baseline: {w_base:.4f}")
print(f"   CASCADE:  {w_casc:.4f}")
print(f"   Delta:    {w_delta:+.4f}")

# Kolmogorov-Smirnov (Proof of Heterogeneity)
# Tests if the CASCADE width distribution is statistically distinct from the Baseline.
ks_stat, ks_p = stats.ks_2samp(df_analysis.base_width, df_analysis.casc_width)

print(f"\n. KOLMOGOROV-SMIRNOV TEST (Interval Heterogeneity)")
print(f"   KS Statistic: {ks_stat:.4f}")
print(f"   p-value:      {ks_p:.4e}")
if ks_p < 0.05:
    print("   Significant. The interval distribution is successfully individualized.")

# Spearman Correlation (Proof of Adaptivity)
# Links width changes directly to the Stage 1 uncertainty signal.
corr, p_corr = stats.spearmanr(df_analysis.casc_width, df_analysis.uncertainty)

print(f"\n. SPEARMAN RANK CORRELATION (Adaptivity R)")
print(f"   Correlation (R): {corr:.4f}")
print(f"   p-value:         {p_corr:.4e}")
if p_corr < 0.05 and corr > 0:
    print("   Significant. Intervals adaptively scale with epistemic uncertainty.")

We compare both Baseline (Standard CP) against both Mondrian and Continous Cascade with 95% Bootstrap

In [ ]:
methods_data = {
    'Baseline (Split CP)': {
        'lower': results_baseline['split']['y_lower'],
        'upper': results_baseline['split']['y_upper']
    },
    'Baseline (CV+)': {
        'lower': results_baseline['cv_plus']['y_lower'],
        'upper': results_baseline['cv_plus']['y_upper']
    },
    'Mondrian (Stratified)': {
        'lower': results_stratified['split']['y_lower'],
        'upper': results_stratified['split']['y_upper']
    },
    'Continuous CASCADE': {
        'lower': results_cascade['split']['y_lower'], 
        'upper': results_cascade['split']['y_upper']
    }
}

y_true_arr = np.array(y_test_2).flatten()
unc_arr = np.array(va_unc_test_2).flatten()

# Bootstrap CIs
print("Running 1000 Bootstraps for 95% Confidence Intervals...\n")
n_boot = 1000
metrics_summary = {}
K_BINS_REPORT = 3 # Defaulting to Tertiles for the global summary

for name, data in methods_data.items():
    boot_cov, boot_wid, boot_rat = [], [], []
    
    for _ in range(n_boot):
        idx = resample(np.arange(len(y_true_arr)), replace=True)
        # Using calculate_metrics to ensure uniform calculation of the Cascade Ratio
        c, w, r = calculate_metrics(y_true_arr[idx], data['lower'][idx], data['upper'][idx], unc_arr[idx], k=K_BINS_REPORT)
        boot_cov.append(c)
        boot_wid.append(w)
        boot_rat.append(r)
        
    metrics_summary[name] = {
        'cov_mean': np.mean(boot_cov), 'cov_ci': (np.percentile(boot_cov, 2.5), np.percentile(boot_cov, 97.5)),
        'wid_mean': np.mean(boot_wid), 'wid_ci': (np.percentile(boot_wid, 2.5), np.percentile(boot_wid, 97.5)),
        'rat_mean': np.mean(boot_rat), 'rat_ci': (np.percentile(boot_rat, 2.5), np.percentile(boot_rat, 97.5))
    }

# Generating Final Table
print(f"{'Method':<25} | {'Coverage (95% CI)':<25} | {'Width (95% CI)':<25} | {'Cascade Ratio (95% CI)'}")
print("-" * 110)

for name, m in metrics_summary.items():
    cov_str = f"{m['cov_mean']:.3f} [{m['cov_ci'][0]:.3f}-{m['cov_ci'][1]:.3f}]"
    wid_str = f"{m['wid_mean']:.3f} [{m['wid_ci'][0]:.3f}-{m['wid_ci'][1]:.3f}]"
    rat_str = f"{m['rat_mean']:.2f} [{m['rat_ci'][0]:.2f}-{m['rat_ci'][1]:.2f}]"
    print(f"{name:<25} | {cov_str:<25} | {wid_str:<25} | {rat_str}")

# Continous CASCADE vs Split Standard CP
print("\n CASCADE vs Split CP Baseline")
# Comparing CR against our official Split CP baseline
w_cont = methods_data['Continuous CASCADE']['upper'] - methods_data['Continuous CASCADE']['lower']
w_base = methods_data['Baseline (Split CP)']['upper'] - methods_data['Baseline (Split CP)']['lower']

diffs_ratio = []
p_low = 100.0 / K_BINS_REPORT
p_high = 100.0 * (K_BINS_REPORT - 1) / K_BINS_REPORT

for _ in range(n_boot):
    idx = resample(np.arange(len(y_true_arr)), replace=True)
    u_b, w_cont_b, w_base_b = unc_arr[idx], w_cont[idx], w_base[idx]
    
    t1, t2 = np.percentile(u_b, [p_low, p_high])
    
    w_low_cont = np.mean(w_cont_b[u_b <= t1])
    r_cont = np.mean(w_cont_b[u_b >= t2]) / w_low_cont if w_low_cont > 0 else 0
    
    w_low_base = np.mean(w_base_b[u_b <= t1])
    r_base = np.mean(w_base_b[u_b >= t2]) / w_low_base if w_low_base > 0 else 0
    
    diffs_ratio.append(r_cont - r_base)

print(f"Mean Difference in CR: {np.mean(diffs_ratio):.3f}")
print(f"95% CI of Difference:  [{np.percentile(diffs_ratio, 2.5):.3f}, {np.percentile(diffs_ratio, 97.5):.3f}]")
print(f"P-Value:               {np.mean(np.array(diffs_ratio) <= 0):.5f}")

# 5. Signal Adaptivity
print("\nStage 1 to Stage 2 Transmission ---")
# Measures how the CP width obeys the Stage 1 Uncertainty signal
corr, p_corr = stats.spearmanr(w_cont, unc_arr)
print(f"Spearman Correlation: {corr:.4f} (p-value: {p_corr:.4e})")
if corr > 0.8:
    print("Strong positive transmission of epistemic uncertainty to CP width.")

### Appendix

Continous Cascade per various groups

In [ ]:
K_BINS = 3 

# Use training set percentiles to maintain  consistency
edges = np.percentile(va_unc_train_2, np.linspace(0, 100, K_BINS + 1))

if K_BINS == 3:
    group_labels = ['Low', 'Medium', 'High']
else:
    group_labels = [f'Bin_{i+1}' for i in range(K_BINS)]

# Cut the test uncertainties using the training edges
categories = pd.cut(u_test, bins=edges, include_lowest=True, labels=group_labels)

df_analyze = pd.DataFrame({
    'Category': categories,
    'True_Y': y_test_2,
    # Continuous CASCADE Results
    'Cont_Lower': results_cascade['split']['y_lower'],
    'Cont_Upper': results_cascade['split']['y_upper'],
    'Cont_Width': results_cascade['split']['y_upper'] - results_cascade['split']['y_lower'],
    # Baseline (Standard Split CP)
    'Base_Lower': results_baseline['split']['y_lower'],
    'Base_Upper': results_baseline['split']['y_upper'],
    'Base_Width': results_baseline['split']['y_upper'] - results_baseline['split']['y_lower']
})

# Generate Table
print(f" Cascade Performance (K={K_BINS}) ---")
print(f"{'Group':<10} | {'N':<4} | {'Method':<15} | {'Coverage':<10} | {'Avg Length':<10} | {'Improvement'}")
print("-" * 80)

for g in group_labels:
    subset = df_analyze[df_analyze['Category'] == g]
    n_patients = len(subset)
    
    if n_patients == 0:
        continue # Skip empty bins if distributions are highly skewed

    # Baseline Stats
    cov_base = np.mean((subset['True_Y'] >= subset['Base_Lower']) & (subset['True_Y'] <= subset['Base_Upper']))
    len_base = subset['Base_Width'].mean()

    # CASCADE Stats
    cov_cont = np.mean((subset['True_Y'] >= subset['Cont_Lower']) & (subset['True_Y'] <= subset['Cont_Upper']))
    len_cont = subset['Cont_Width'].mean()

    # Improvement Calculation
    diff_pct = (len_cont - len_base) / len_base * 100

    print(f"{g:<10} | {n_patients:<4} | {'Baseline':<15} | {cov_base:<10.2%} | {len_base:<10.3f} | -")
    print(f"{g:<10} | {n_patients:<4} | {'CASCADE (Ours)':<15} | {cov_cont:<10.2%} | {len_cont:<10.3f} | {diff_pct:+.1f}%")
    print("-" * 80)

Evaluating gains

In [ ]:
K_BINS_REPORT = 3

print(f" fficiency Gain for Confident Patients (K={K_BINS_REPORT}) ---")

# 1. Pull the finalized widths directly from the source dictionaries
w_cont = results_cascade['split']['y_upper'] - results_cascade['split']['y_lower']
w_base = results_baseline['split']['y_upper'] - results_baseline['split']['y_lower']
avg_base_width = np.mean(w_base) 

# 2. Isolate the "Confident Patients" 
# We use the Training Set percentiles to perfectly match the K=3 Table logic
threshold_low = np.percentile(va_unc_train_2, 100.0 / K_BINS_REPORT)
mask_confident = (va_unc_test_2 <= threshold_low).flatten()

# 3. Calculate the gain
avg_width_ours = np.mean(w_cont[mask_confident])
reduction_abs = avg_base_width - avg_width_ours
reduction_pct = (reduction_abs / avg_base_width) * 100

print(f"{'Metric':<35} | {'Value':<10}")
print("-" * 50)
print(f"{'Global Baseline Width (Status Quo)':<35} | {avg_base_width:.3f}")
print(f"{'Confident Patient Width (CASCADE)':<35} | {avg_width_ours:.3f}")
print(f"{'Absolute Reduction':<35} | {reduction_abs:.3f}")
print(f"{'Precision Improvement (%)':<35} | {reduction_pct:.1f}%")

Coverage, Length and Cascade Ratio for different K (Continous Cascade)

In [ ]:
k_list = [3, 5, 7]
appendix_results = []

y_t = np.array(y_test_2).flatten()
u_t = np.array(va_unc_test_2).flatten()
u_train = np.array(va_unc_train_2).flatten() # <-- SYNCED: Using Train Set
w_c = results_cascade['split']['y_upper'] - results_cascade['split']['y_lower']

for k in k_list:
    p_low = 100.0 / k
    p_high = 100.0 * (k - 1) / k
    
    # Calculate thresholds dynamically on the train set (matches tuning loop)
    t_low = np.percentile(u_train, p_low)
    t_high = np.percentile(u_train, p_high)
    
    # Masks exactly as evaluated in tuning (applied to Test set)
    mask_low = u_t <= t_low
    mask_high = u_t > t_high
    
    w_low = np.mean(w_c[mask_low])
    w_high = np.mean(w_c[mask_high])
    
    # Calculate LOCAL coverage for these extremes
    cov_low = np.mean((y_t[mask_low] >= results_cascade['split']['y_lower'][mask_low]) & 
                      (y_t[mask_low] <= results_cascade['split']['y_upper'][mask_low]))
    
    cov_high = np.mean((y_t[mask_high] >= results_cascade['split']['y_lower'][mask_high]) & 
                       (y_t[mask_high] <= results_cascade['split']['y_upper'][mask_high]))
    
    # Calculate the exact CR
    cr = w_high / w_low if w_low > 1e-6 else 0.0
    
    appendix_results.append({
        'Resolution (K)': k,
        'Low Cov': f"{cov_low:.2%}",
        'Low Len': w_low,
        'High Cov': f"{cov_high:.2%}",
        'High Len': w_high,
        'Cascade Ratio': cr
    })

df_appendix = pd.DataFrame(appendix_results)
display(df_appendix.round(2))

Coverage, Length and Cascade Ratio for different K (Mondrian)

In [ ]:
K_BINS_REPORT = 3 #

print(f" APPENDIX TABLE (Mondrian): K={K_BINS_REPORT} ")

y_t = np.array(y_test_2).flatten()
u_t = np.array(va_unc_test_2).flatten()
u_train = np.array(va_unc_train_2).flatten()

edges = np.percentile(u_train, np.linspace(0, 100, K_BINS_REPORT + 1))

# Dynamically assign names based on K
if K_BINS_REPORT == 3:
    group_labels = ['Low', 'Medium', 'High']
else:
    group_labels = [f'Bin_{i+1}' for i in range(K_BINS_REPORT)]

bins = pd.cut(u_t, bins=edges, include_lowest=True, labels=group_labels)

# ========================================================
# APPENDIX TABLE 1: Baseline vs Mondrian CASCADE
# ========================================================
print(f"\n--- TABLE 1: BASELINE VS MONDRIAN CASCADE (K={K_BINS_REPORT}) ---")
table_mondrian = []

base_low = results_baseline['split']['y_lower'].flatten()
base_high = results_baseline['split']['y_upper'].flatten()

mondrian_low = results_stratified['split']['y_lower'].flatten()
mondrian_high = results_stratified['split']['y_upper'].flatten()

for group in group_labels:
    mask = (bins == group)
    
    # Only calculate if the bin has patients
    if np.sum(mask) > 0:
        cov_b = np.mean((y_t[mask] >= base_low[mask]) & (y_t[mask] <= base_high[mask]))
        len_b = np.mean(base_high[mask] - base_low[mask])
        
        cov_m = np.mean((y_t[mask] >= mondrian_low[mask]) & (y_t[mask] <= mondrian_high[mask]))
        len_m = np.mean(mondrian_high[mask] - mondrian_low[mask])
        
        table_mondrian.append({
            'Uncertainty Bin': group,
            'Baseline Cov': f"{cov_b*100:.1f}%",
            'Baseline Len': f"{len_b:.3f}",
            'Mondrian Cov': f"{cov_m*100:.1f}%",
            'Mondrian Len': f"{len_m:.3f}"
        })

pd.DataFrame(table_mondrian)

Full Beta Ablation

In [ ]:
print(f"\nFull Beta Ablation (K={K_BINS_REPORT}) ---")

betas_to_test = np.arange(0.0, 1.6, 0.1)
global_mean_unc = np.mean(va_unc_proper)
y_pred_calib_base = xgb_model_proper.predict(X_calib)
residuals_calib_base = np.abs(y_calib - y_pred_calib_base)
y_pred_test_base = xgb_model_proper.predict(X_test_2)

def get_sigma(unc_array, beta):
    norm_diff = (unc_array / global_mean_unc) - 1.0
    sigma = 1.0 + (beta * norm_diff)
    return np.clip(sigma, a_min=0.2, a_max=5.0)

table_ablation = []

for b in betas_to_test:
    sigma_calib = get_sigma(np.array(va_unc_calib).flatten(), b)
    scores = residuals_calib_base / sigma_calib
    
    n_calib = len(scores)
    q_level = np.ceil((n_calib + 1) * (1 - ALPHA)) / n_calib
    q_hat = np.quantile(scores, min(1.0, q_level), method='higher')
    
    sigma_test = get_sigma(u_t, b)
    half_width = q_hat * sigma_test
    casc_low = y_pred_test_base - half_width
    casc_high = y_pred_test_base + half_width
    
    row_data = {'Beta': b}
    
    for group in group_labels:
        mask = (bins == group)
        if np.sum(mask) > 0:
            cov = np.mean((y_t[mask] >= casc_low[mask]) & (y_t[mask] <= casc_high[mask]))
            avg_len = np.mean(casc_high[mask] - casc_low[mask])
            
            row_data[f'{group} Cov'] = f"{cov*100:.1f}%"
            row_data[f'{group} Len'] = f"{avg_len:.3f}"
        
    table_ablation.append(row_data)

df_ablation = pd.DataFrame(table_ablation)

cols = ['Beta']
for group in group_labels:
    cols.extend([f'{group} Cov', f'{group} Len'])
    
df_ablation[cols]